In [1]:
import sqlite3
import pandas as pd

In [2]:
###1.1 Load the Data

conn = sqlite3.connect("data/data.db")
raw = pd.read_sql("SELECT * FROM ohlc_hw", conn)
#conn.close()

df = raw.copy()

print(df.shape)
df.head()

(397711, 8)


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [3]:
#Display Basic statistics
df.describe()

,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


In [4]:
#Identify columns
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 31.6 MB


In [5]:
df.describe(include="all")

,ts,open,high,low,close,adj_close,volume,ticker
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05,397711
unique,15868,NaN,NaN,NaN,NaN,NaN,NaN,150
top,2020-11-24,NaN,NaN,NaN,NaN,NaN,NaN,PG
freq,40,NaN,NaN,NaN,NaN,NaN,NaN,15450
mean,NaN,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07,NaN
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08,NaN
min,NaN,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00,NaN
25%,NaN,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06,NaN
50%,NaN,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06,NaN
75%,NaN,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07,NaN


In [6]:
#date range
df["ts"].max()

'2024-12-06'

In [7]:
df["ts"].min()

'1962-01-02'

In [8]:
#list of tickers
df["ticker"].unique()

#The raw ticker column contains both plain stock symbols and HTML-formatted strings.

<ArrowStringArray>
[                                                                           'CAT',
                                                                            'CVX',
                                                                             'HD',
                                                                           'AAPL',
                                                                            'TRV',
                                                                            'JNJ',
                                                                            'DIS',
                   '<a target="_blank"><span style="color: blue">CSCO</span></a>',
                                                                            'SHW',
                                                                             'KO',
 ...
  '<a target="_blank"><span style="font-weight: bold; color: red">IBM</span></a>',
                    '<a target="_blank"><span style="color: red

In [9]:
df["ticker"] = (
    df["ticker"]
    .str.replace(r"<[^>]*>", "", regex=True)
    .str.strip()
)

In [10]:
df["ticker"].unique()

<ArrowStringArray>
[ 'CAT',  'CVX',   'HD', 'AAPL',  'TRV',  'JNJ',  'DIS', 'CSCO',  'SHW',
   'KO',   'VZ',  'IBM',   'PG',  'HON',  'AXP',  'MMM',   'GS',   'BA',
  'CRM',  'UNH',  'NKE', 'NVDA', 'AMGN',  'MCD',    'V',  'MRK',  'WMT',
  'JPM', 'MSFT', 'AMZN']
Length: 30, dtype: str

In [11]:
###1.2 Dectect Data Quality Issue
# Missing Data
df.isna().sum()

#adj_close is missing 19855 values

ts               0
open             0
high             0
low              0
close            0
adj_close    19855
volume           0
ticker           0
dtype: int64

In [12]:
##check what's the data type of ts. It's a string so convert it to datetime.
df["ts"].head()

0    1975-04-14
1    1978-04-05
2    2018-05-25
3    1995-12-07
4    1981-07-23
Name: ts, dtype: str

In [13]:
df["ts"] = pd.to_datetime(df["ts"])

In [14]:
df["ts"].head()

0   1975-04-14
1   1978-04-05
2   2018-05-25
3   1995-12-07
4   1981-07-23
Name: ts, dtype: datetime64[us]

In [15]:
all_dates = sorted(df["ts"].unique())

In [16]:
len(all_dates)

15868

In [17]:
all_dates

[Timestamp('1962-01-02 00:00:00'),
 Timestamp('1962-01-03 00:00:00'),
 Timestamp('1962-01-04 00:00:00'),
 Timestamp('1962-01-05 00:00:00'),
 Timestamp('1962-01-08 00:00:00'),
 Timestamp('1962-01-09 00:00:00'),
 Timestamp('1962-01-10 00:00:00'),
 Timestamp('1962-01-11 00:00:00'),
 Timestamp('1962-01-12 00:00:00'),
 Timestamp('1962-01-15 00:00:00'),
 Timestamp('1962-01-16 00:00:00'),
 Timestamp('1962-01-17 00:00:00'),
 Timestamp('1962-01-18 00:00:00'),
 Timestamp('1962-01-19 00:00:00'),
 Timestamp('1962-01-22 00:00:00'),
 Timestamp('1962-01-23 00:00:00'),
 Timestamp('1962-01-24 00:00:00'),
 Timestamp('1962-01-25 00:00:00'),
 Timestamp('1962-01-26 00:00:00'),
 Timestamp('1962-01-29 00:00:00'),
 Timestamp('1962-01-30 00:00:00'),
 Timestamp('1962-01-31 00:00:00'),
 Timestamp('1962-02-01 00:00:00'),
 Timestamp('1962-02-02 00:00:00'),
 Timestamp('1962-02-05 00:00:00'),
 Timestamp('1962-02-06 00:00:00'),
 Timestamp('1962-02-07 00:00:00'),
 Timestamp('1962-02-08 00:00:00'),
 Timestamp('1962-02-

In [18]:
df["ticker"].unique()

<ArrowStringArray>
[ 'CAT',  'CVX',   'HD', 'AAPL',  'TRV',  'JNJ',  'DIS', 'CSCO',  'SHW',
   'KO',   'VZ',  'IBM',   'PG',  'HON',  'AXP',  'MMM',   'GS',   'BA',
  'CRM',  'UNH',  'NKE', 'NVDA', 'AMGN',  'MCD',    'V',  'MRK',  'WMT',
  'JPM', 'MSFT', 'AMZN']
Length: 30, dtype: str

In [19]:
df["ticker"] == "AAPL"

0         False
1         False
2         False
3         False
4          True
          ...  
397706    False
397707    False
397708    False
397709    False
397710    False
Name: ticker, Length: 397711, dtype: bool

In [20]:
df.head()

,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [21]:
df.iloc[4,7]

'AAPL'

In [22]:
AAPL_dates =sorted(df.loc[df["ticker"] =="AAPL","ts"].unique())

In [23]:
all_dates
AAPL_dates

[Timestamp('1980-12-12 00:00:00'),
 Timestamp('1980-12-15 00:00:00'),
 Timestamp('1980-12-16 00:00:00'),
 Timestamp('1980-12-17 00:00:00'),
 Timestamp('1980-12-18 00:00:00'),
 Timestamp('1980-12-19 00:00:00'),
 Timestamp('1980-12-22 00:00:00'),
 Timestamp('1980-12-23 00:00:00'),
 Timestamp('1980-12-24 00:00:00'),
 Timestamp('1980-12-26 00:00:00'),
 Timestamp('1980-12-29 00:00:00'),
 Timestamp('1980-12-30 00:00:00'),
 Timestamp('1980-12-31 00:00:00'),
 Timestamp('1981-01-02 00:00:00'),
 Timestamp('1981-01-05 00:00:00'),
 Timestamp('1981-01-06 00:00:00'),
 Timestamp('1981-01-07 00:00:00'),
 Timestamp('1981-01-08 00:00:00'),
 Timestamp('1981-01-09 00:00:00'),
 Timestamp('1981-01-12 00:00:00'),
 Timestamp('1981-01-13 00:00:00'),
 Timestamp('1981-01-14 00:00:00'),
 Timestamp('1981-01-15 00:00:00'),
 Timestamp('1981-01-16 00:00:00'),
 Timestamp('1981-01-19 00:00:00'),
 Timestamp('1981-01-20 00:00:00'),
 Timestamp('1981-01-21 00:00:00'),
 Timestamp('1981-01-22 00:00:00'),
 Timestamp('1981-01-

In [24]:
AAPL_dates[0]

Timestamp('1980-12-12 00:00:00')

In [25]:
AAPL_dates[-1]

Timestamp('2024-10-18 00:00:00')

In [26]:
all_dates[0]

Timestamp('1962-01-02 00:00:00')

In [27]:
all_dates[-1]

Timestamp('2024-12-06 00:00:00')

In [28]:
expected_dates = []

first_date = AAPL_dates[0]
last_date = AAPL_dates[-1]

for date in all_dates:
    if first_date <= date <= last_date:
        expected_dates.append(date)

print(expected_dates)

[Timestamp('1980-12-12 00:00:00'), Timestamp('1980-12-15 00:00:00'), Timestamp('1980-12-16 00:00:00'), Timestamp('1980-12-17 00:00:00'), Timestamp('1980-12-18 00:00:00'), Timestamp('1980-12-19 00:00:00'), Timestamp('1980-12-22 00:00:00'), Timestamp('1980-12-23 00:00:00'), Timestamp('1980-12-24 00:00:00'), Timestamp('1980-12-26 00:00:00'), Timestamp('1980-12-29 00:00:00'), Timestamp('1980-12-30 00:00:00'), Timestamp('1980-12-31 00:00:00'), Timestamp('1981-01-02 00:00:00'), Timestamp('1981-01-05 00:00:00'), Timestamp('1981-01-06 00:00:00'), Timestamp('1981-01-07 00:00:00'), Timestamp('1981-01-08 00:00:00'), Timestamp('1981-01-09 00:00:00'), Timestamp('1981-01-12 00:00:00'), Timestamp('1981-01-13 00:00:00'), Timestamp('1981-01-14 00:00:00'), Timestamp('1981-01-15 00:00:00'), Timestamp('1981-01-16 00:00:00'), Timestamp('1981-01-19 00:00:00'), Timestamp('1981-01-20 00:00:00'), Timestamp('1981-01-21 00:00:00'), Timestamp('1981-01-22 00:00:00'), Timestamp('1981-01-23 00:00:00'), Timestamp('19

In [29]:
sorted(set(expected_dates) - set(AAPL_dates))

[Timestamp('1981-08-10 00:00:00'),
 Timestamp('1983-09-23 00:00:00'),
 Timestamp('1989-01-17 00:00:00'),
 Timestamp('1989-01-27 00:00:00'),
 Timestamp('1989-02-14 00:00:00'),
 Timestamp('1989-12-12 00:00:00')]

In [30]:

print(len(all_dates))       # how many dates in the full calendar
print(len(expected_dates))  # how many fall inside AAPL's range
print(len(AAPL_dates))      # how many AAPL actually has

15868
11056
11050


In [31]:
all_tickers = df["ticker"].unique()
result = []

for t in all_tickers:
     ticker_dates =sorted(df.loc[df["ticker"] == t,"ts"].unique())
     first_date = ticker_dates[0]
     last_date = ticker_dates[-1]
     expected_dates = []
     for date in all_dates:
         if first_date <= date <= last_date:
             expected_dates.append(date)
     missing_dates= sorted(set(expected_dates) - set(ticker_dates))
     result.append([t, first_date, last_date, len(missing_dates), missing_dates])


In [32]:
missing_days_df = pd.DataFrame(result)

In [33]:
missing_days_df.head()


,0,1,2,3,4
0,CAT,1962-01-02,2024-10-18,26,"[1974-01-11 00:00:00, 1974-01-18 00:00:00, 197..."
1,CVX,1962-01-02,2024-12-06,25,"[1974-01-11 00:00:00, 1974-01-18 00:00:00, 197..."
2,HD,1981-09-22,2024-10-18,0,[]
3,AAPL,1980-12-12,2024-10-18,6,"[1981-08-10 00:00:00, 1983-09-23 00:00:00, 198..."
4,TRV,1972-04-03,2024-10-18,28,"[1972-12-14 00:00:00, 1973-03-01 00:00:00, 197..."


In [34]:
range(len(missing_days_df))

range(0, 30)

In [35]:
##Create a summary: "Ticker X is missing Y trading days between [start] and [end]"

for i in range(len(missing_days_df)):
    ticker = missing_days_df.iloc[i, 0]
    days = missing_days_df.iloc[i, 3]
    missing_list = missing_days_df.iloc[i, 4]

    if days > 0:
        start = min(missing_list)
        end = max(missing_list)
        print(f"Ticker {ticker} is missing {days} trading days between {start} and {end}")

Ticker CAT is missing 26 trading days between 1974-01-11 00:00:00 and 1985-09-27 00:00:00
Ticker CVX is missing 25 trading days between 1974-01-11 00:00:00 and 1975-04-25 00:00:00
Ticker AAPL is missing 6 trading days between 1981-08-10 00:00:00 and 1989-12-12 00:00:00
Ticker TRV is missing 28 trading days between 1972-12-14 00:00:00 and 1975-08-04 00:00:00
Ticker JNJ is missing 26 trading days between 1974-01-11 00:00:00 and 1985-09-27 00:00:00
Ticker DIS is missing 1 trading days between 1985-09-27 00:00:00 and 1985-09-27 00:00:00
Ticker SHW is missing 26 trading days between 1974-01-11 00:00:00 and 1985-09-27 00:00:00
Ticker KO is missing 26 trading days between 1974-01-11 00:00:00 and 1985-09-27 00:00:00
Ticker VZ is missing 1 trading days between 1985-09-27 00:00:00 and 1985-09-27 00:00:00
Ticker IBM is missing 25 trading days between 1974-01-11 00:00:00 and 1975-04-25 00:00:00
Ticker PG is missing 25 trading days between 1974-01-11 00:00:00 and 1975-04-25 00:00:00
Ticker HON is m

In [36]:
##Outliers:

#Calculate daily returns: (close - prev_close) / prev_close
print(df.columns)

Index(['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker'],
      dtype='str')


In [37]:
aapl = df.loc[df["ticker"] == "AAPL"].sort_values("ts").copy()aapl.head()


aapl["prev_close"] = aapl["close"].shift(1)
aapl["return"] = (aapl["close"] - aapl["prev_close"]) / aapl["prev_close"]

aapl[["ts", "close", "prev_close", "return"]].head()

,ts,close,prev_close,return
72412,1980-12-12,0.12835,NaN,NaN
102310,1980-12-15,0.12165,0.12835,-0.052201
294188,1980-12-16,0.11272,0.12165,-0.073407
285108,1980-12-17,0.11551,0.11272,0.024752
194322,1980-12-18,0.11886,0.11551,0.029002


In [38]:
df = df.sort_values(["ticker", "ts"])

df["prev_close"] = df.groupby("ticker")["close"].shift(1)
df["return"] = (df["close"] - df["prev_close"]) / df["prev_close"]

In [39]:
df[["ticker","ts", "close", "prev_close", "return"]].groupby("ticker").head()

,ticker,ts,close,prev_close,return
72412,AAPL,1980-12-12,0.128350,NaN,NaN
102310,AAPL,1980-12-15,0.121650,0.128350,-0.052201
294188,AAPL,1980-12-16,0.112720,0.121650,-0.073407
285108,AAPL,1980-12-17,0.115510,0.112720,0.024752
194322,AAPL,1980-12-18,0.118860,0.115510,0.029002
...,...,...,...,...,...
102868,WMT,1972-04-03,0.017578,NaN,NaN
135404,WMT,1972-04-04,0.017415,0.017578,-0.009273
244993,WMT,1972-04-04,0.017415,0.017415,0.000000
179444,WMT,1972-04-05,0.016927,0.017415,-0.028022


In [40]:
df = df.reset_index(drop=True)
print(df.index.is_unique)

True


In [41]:
##Flag abs(return - mean_return) > 3 * std_return within each ticker, using the sample standard deviation (ddof=1); flag both tails. Exclude undefined returns and treat zero/undefined standard deviation explicitly. This retrospective screen is a review flag, not a deletion rule.
outliers = []
skipped = []

for t in df["ticker"].unique():
    ticker_data = df.loc[df["ticker"] == t]

    mean_return = ticker_data["return"].mean()
    std_return = ticker_data["return"].std(ddof=1)

    if pd.isna(std_return) or std_return == 0:
        skipped.append([t, std_return])
    else:
        for i in range(len(ticker_data)):
            r = ticker_data["return"].iloc[i]
            if not pd.isna(r) and abs(r - mean_return) > 3 * std_return:
                outliers.append([ticker_data.index[i], t, ticker_data["ts"].iloc[i], r, mean_return, std_return])

outliers_df = pd.DataFrame(outliers, columns=["row", "ticker", "ts", "return", "mean_return", "std_return"])
skipped_df = pd.DataFrame(skipped, columns=["ticker", "std_return"])

In [42]:
df["return_outlier"] = False
df.loc[outliers_df["row"], "return_outlier"] = True

In [43]:
print(df["return_outlier"].sum())
print(len(outliers_df))

5741
5741


In [44]:
df.head()

,ts,open,high,low,close,adj_close,volume,ticker,prev_close,return,return_outlier
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,NaN,NaN,False
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,0.12835,-0.052201,False
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,0.12165,-0.073407,False
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,0.11272,0.024752,False
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,0.11551,0.029002,False


In [45]:
df["bad_high_low"] = (df["high"] < df["close"]) | (df["low"] > df["close"])

In [46]:
print(df["bad_high_low"].sum())
df.loc[df["bad_high_low"], ["ticker", "ts", "high", "low", "close"]]

0


,ticker,ts,high,low,close


In [47]:
import numpy as np

price_cols = ["open", "high", "low", "close", "adj_close"]

df["bad_price"] = False

for col in price_cols:
    df.loc[df[col] <= 0, "bad_price"] = True
    df.loc[df[col] == np.inf, "bad_price"] = True
    df.loc[df[col] == -np.inf, "bad_price"] = True

In [48]:
print(df["bad_price"].sum())
df.loc[df["bad_price"], ["ticker", "ts"] + price_cols]

5


,ticker,ts,open,high,low,close,adj_close
93926,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750
181061,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012
230450,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090
263321,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447
309937,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300


In [49]:
##Inconsistencies: Verify that low <= open <= high and low <= close <= high. 
##Check for zero volume days (could indicate stale data)
df["bad_bounds"] = False

df.loc[df["open"] < df["low"], "bad_bounds"] = True
df.loc[df["open"] > df["high"], "bad_bounds"] = True
df.loc[df["close"] < df["low"], "bad_bounds"] = True
df.loc[df["close"] > df["high"], "bad_bounds"] = True

In [50]:
print(df["bad_bounds"].sum())
df.loc[df["bad_bounds"], ["ticker", "ts", "open", "high", "low", "close"]]

5


,ticker,ts,open,high,low,close
93926,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000
181061,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338
230450,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955
263321,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537
309937,PG,1962-01-02,0.0,1.44000,1.42000,1.43000


In [51]:
df["zero_volume"] = False

df.loc[df["volume"] == 0, "zero_volume"] = True

In [52]:
print(df["zero_volume"].sum())
df.loc[df["zero_volume"], ["ticker", "ts", "close", "volume"]]

81


,ticker,ts,close,volume
30876,AXP,1972-12-14,4.010000,0
30877,AXP,1972-12-14,4.010000,0
152443,HON,1982-09-23,5.540000,0
152444,HON,1982-09-24,5.540000,0
167090,IBM,1973-10-09,15.640000,0
...,...,...,...,...
384197,WMT,1975-02-20,0.009684,0
384248,WMT,1975-05-01,0.011149,0
384326,WMT,1975-08-13,0.015462,0
385296,WMT,1979-03-15,0.031738,0


In [53]:
df.loc[df["zero_volume"], "ticker"].value_counts()

ticker
WMT    55
JNJ    13
IBM     3
TRV     3
AXP     2
HON     2
MRK     1
NKE     1
SHW     1
Name: count, dtype: int64

In [54]:
#Create a Data Quality Report DataFrame with columns:
df.head()

,ts,open,high,low,close,adj_close,volume,ticker,prev_close,return,return_outlier,bad_high_low,bad_price,bad_bounds,zero_volume
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,NaN,NaN,False,False,False,False,False
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,0.12835,-0.052201,False,False,False,False,False
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,0.12165,-0.073407,False,False,False,False,False
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,0.11272,0.024752,False,False,False,False,False
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,0.11551,0.029002,False,False,False,False,False


In [55]:
issues = [
    ["return_outlier", "Return more than 3 std from mean", "medium"],
    ["bad_high_low", "High below close or low above close", "high"],
    ["bad_price", "Price zero, negative, or infinite", "high"],
    ["bad_bounds", "Open or close outside low-high range", "high"],
    ["zero_volume", "Volume is zero", "low"],
]

report = []

for item in issues:
    flagged = df.loc[df[item[0]]]
    for i in range(len(flagged)):
        report.append([flagged["ticker"].iloc[i], item[0], flagged["ts"].iloc[i], item[1], item[2]])

report_df = pd.DataFrame(report, columns=["ticker", "issue_type", "issue_date", "description", "severity"])
report_df

,ticker,issue_type,issue_date,description,severity
0,AAPL,return_outlier,1980-12-26,Return more than 3 std from mean,medium
1,AAPL,return_outlier,1981-04-02,Return more than 3 std from mean,medium
2,AAPL,return_outlier,1981-09-25,Return more than 3 std from mean,medium
3,AAPL,return_outlier,1981-10-02,Return more than 3 std from mean,medium
4,AAPL,return_outlier,1981-12-17,Return more than 3 std from mean,medium
...,...,...,...,...,...
5827,WMT,zero_volume,1975-02-20,Volume is zero,low
5828,WMT,zero_volume,1975-05-01,Volume is zero,low
5829,WMT,zero_volume,1975-08-13,Volume is zero,low
5830,WMT,zero_volume,1979-03-15,Volume is zero,low


In [56]:
###1.3 Clean the Data
##Handle Missing Values

#For missing prices: Use forward-fill (carry last known price)
raw_cols = ["open", "high", "low", "close"]
clean = df.copy()
clean["price_filled"] = clean[raw_cols].isna().any(axis=1)
clean[raw_cols] = clean.groupby("ticker")[raw_cols].ffill()



In [57]:
print("Rows filled:", clean["price_filled"].sum())
print("Still missing:", clean[raw_cols].isna().sum())

Rows filled: 0
Still missing: open     0
high     0
low      0
close    0
dtype: int64


In [58]:
#For missing volume: Fill with 0 and keep a repair flag (unknown volume is not evidence of no trading)
clean["volume_filled"] = clean["volume"].isna()
clean["volume"] = clean["volume"].fillna(0)

print("Rows with filled volume:", clean["volume_filled"].sum())

Rows with filled volume: 0


In [59]:
print("Still missing:", clean["volume"].isna().sum())

Still missing: 0


In [60]:
#Drop rows where ALL raw price columns (open, high, low, close) are missing before filling
#there are none

In [61]:
ref = pd.read_sql("SELECT ticker, ts, adj_close / close AS factor FROM ohlc", conn)

clean["ts"] = pd.to_datetime(clean["ts"]).dt.strftime("%Y-%m-%d")
ref["ts"] = pd.to_datetime(ref["ts"]).dt.strftime("%Y-%m-%d")

clean = clean.drop(columns=["factor"], errors="ignore")
clean = clean.merge(ref, on=["ticker", "ts"], how="left")

# Fill missing adj_close
fix = clean["adj_close"].isna() & ~clean["price_filled"]
clean.loc[fix, "adj_close"] = clean["close"] * clean["factor"]
clean["adj_from_ref"] = fix & clean["adj_close"].notna()

quarantine = clean.loc[clean["adj_close"].isna()]
clean = clean.loc[clean["adj_close"].notna()]

# Fix adj_close
ratio = clean["adj_close"] / clean["close"]
typical = ratio.groupby(clean["ticker"]).transform("median")

off_ref = (ratio / clean["factor"] - 1).abs() > 0.05
off_typical = clean["factor"].isna() & ((ratio / typical > 2) | (ratio / typical < 0.5))

clean.loc[off_ref, "adj_close"] = clean["close"] * clean["factor"]
clean["adj_repaired"] = off_ref

quarantine = pd.concat([quarantine, clean.loc[off_typical]])
clean = clean.loc[~off_typical]

print("Rebuilt:", clean["adj_from_ref"].sum())
print("Repaired:", clean["adj_repaired"].sum())
print("Quarantined:", len(quarantine), "Retained:", len(clean))

Rebuilt: 19855
Repaired: 11265
Quarantined: 0 Retained: 397711


In [62]:
print("Missing adj_close:", clean["adj_close"].isna().sum())

Missing adj_close: 0


In [63]:
##Handle Outliers:

#For impossible price relationships (high < close, etc.).Use the previous day's close
df.head()

,ts,open,high,low,close,adj_close,volume,ticker,prev_close,return,return_outlier,bad_high_low,bad_price,bad_bounds,zero_volume
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,NaN,NaN,False,False,False,False,False
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,0.12835,-0.052201,False,False,False,False,False
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,0.12165,-0.073407,False,False,False,False,False
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,0.11272,0.024752,False,False,False,False,False
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,0.11551,0.029002,False,False,False,False,False


In [64]:
bad = clean["bad_high_low"] | clean["bad_bounds"]
prev = clean.groupby("ticker")[["close", "adj_close"]].shift(1)

for col in ["open", "high", "low", "close"]:
    clean.loc[bad, col] = prev["close"]
clean.loc[bad, "adj_close"] = prev["adj_close"]
clean["bounds_repaired"] = bad

quarantine = pd.concat([quarantine, clean.loc[clean["close"].isna()]])
clean = clean.loc[clean["close"].notna()]

In [65]:
clean.head()

,ts,open,high,low,close,adj_close,volume,ticker,prev_close,return,...,bad_high_low,bad_price,bad_bounds,zero_volume,price_filled,volume_filled,factor,adj_from_ref,adj_repaired,bounds_repaired
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,NaN,NaN,...,False,False,False,False,False,False,0.770866,False,False,False
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,0.12835,-0.052201,...,False,False,False,False,False,False,0.770891,False,False,False
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,0.12165,-0.073407,...,False,False,False,False,False,False,0.770899,False,False,False
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,0.11272,0.024752,...,False,False,False,False,False,False,0.770898,False,False,False
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,0.11551,0.029002,...,False,False,False,False,False,False,0.770885,False,False,False


In [66]:
##For extreme returns (>3 std dev):Investigate: Is it a stock split? Merger? Data error?If data error: interpolate using neighboring daysIf valid (e.g., major news): keep it but flag it
cols = ["open", "high", "low", "close", "adj_close"]
prev = clean.groupby("ticker")[cols + ["factor"]].shift(1)
nxt = clean.groupby("ticker")[cols].shift(-1)

split = clean["return_outlier"] & ((clean["factor"] / prev["factor"] - 1).abs() > 0.01)
error = clean["return_outlier"] & ~split & ((nxt["close"] / prev["close"] - 1).abs() < 0.02)

clean["return_flag"] = ""
clean.loc[clean["return_outlier"], "return_flag"] = "kept, cause unverified"
clean.loc[split, "return_flag"] = "split or corporate action"
clean.loc[error, "return_flag"] = "data error, interpolated"

for col in cols:
    clean.loc[error, col] = (prev[col] + nxt[col]) / 2

In [67]:
print(clean["return_flag"].value_counts())
print("high < close:", (clean["high"] < clean["close"]).sum())
print("low > close:", (clean["low"] > clean["close"]).sum())

return_flag
                             391965
kept, cause unverified         5393
data error, interpolated        321
split or corporate action        27
Name: count, dtype: int64
high < close: 0
low > close: 0


In [68]:
###Validation:
##Create a new table ohlc_hw_cleaned in the database
##Verify: No missing, non-finite, or nonpositive values in retained raw and adjusted price columns; unique (ticker, ts) keys; finite nonnegative volume
##Verify: All price relationships are valid
##Show before/after statistics
import numpy as np

prices = ["open", "high", "low", "close", "adj_close"]

# Remove duplicate (ticker, ts) rows, keep first
dupes = clean.duplicated(["ticker", "ts"])
removed = dupes.sum()
clean = clean.loc[~dupes]

# Verify
print("Missing prices:", clean[prices].isna().sum().sum())
print("Infinite prices:", np.isinf(clean[prices]).sum().sum())
print("Nonpositive prices:", (clean[prices] <= 0).sum().sum())
print("Duplicate keys:", clean.duplicated(["ticker", "ts"]).sum())
print("Bad volume:", (clean["volume"].isna() | np.isinf(clean["volume"]) | (clean["volume"] < 0)).sum())
print("high < close:", (clean["high"] < clean["close"]).sum())
print("low > close:", (clean["low"] > clean["close"]).sum())
print("open outside range:", ((clean["open"] < clean["low"]) | (clean["open"] > clean["high"])).sum())

# Row accounting
print("Raw:", len(df), "Retained:", len(clean), "Removed:", removed, "Quarantined:", len(quarantine))
print("Adds up:", len(df) == len(clean) + removed + len(quarantine))

# Save
clean.to_sql("ohlc_hw_cleaned", conn, if_exists="replace", index=False)
quarantine.to_sql("ohlc_hw_quarantine", conn, if_exists="replace", index=False)

Missing prices: 0
Infinite prices: 0
Nonpositive prices: 0
Duplicate keys: 0
Bad volume: 0
high < close: 0
low > close: 0
open outside range: 0
Raw: 397711 Retained: 368246 Removed: 29460 Quarantined: 5
Adds up: True


5

In [69]:
pd.DataFrame({
    "before": df["close"].describe(),
    "after": clean["close"].describe()
})

,before,after
count,397711.000000,368246.000000
mean,46.292903,46.317688
std,67.141879,67.155113
min,0.005208,0.005208
25%,4.200000,4.210000
50%,19.860000,19.910000
75%,57.530000,57.580000
max,605.400000,605.400000


In [70]:
pd.read_sql("SELECT COUNT(*) FROM ohlc_hw_cleaned", conn)

,COUNT(*)
0,368246


In [71]:
conn.close()